# 교육·학군 변수 — 데이터 수집 & 전처리 파이프라인

## 개요

아파트 실거래가 예측 모델에 투입되는 **교육·학군 변수 11개**의 수집 출처, 수집 방법, 전처리 과정을 단계별로 기술합니다.

| 변수명 | 설명 | 출처 |
|---|---|---|
| `elem_cnt_500m` | 반경 500m 이내 초등학교 수 | 카카오 로컬 API |
| `mid_cnt_500m` | 반경 500m 이내 중학교 수 | 카카오 로컬 API |
| `high_cnt_500m` | 반경 500m 이내 고등학교 수 | 카카오 로컬 API |
| `has_elem_500m` | 초등학교 접근 더미 (1=있음) | 파생 |
| `has_mid_500m` | 중학교 접근 더미 (1=있음) | 파생 |
| `has_high_500m` | 고등학교 접근 더미 (1=있음) | 파생 |
| `elem_nearest_m` | 최근접 초등학교 거리 (m), 없으면 999m 패널티 | 카카오 로컬 API |
| `log_elem_nearest_m` | 위의 로그 변환값 | 파생 |
| `elem_access_score` | 초등학교 접근성 점수 (가까울수록 높음) | 파생 |
| `school_density_index` | 학교 밀도 종합 인덱스 (초×3 + 중×2 + 고×1) | 파생 |
| `academy_cnt_500m_t` | 반경 500m 이내 학원 수 | 소상공인시장진흥공단 |

### 데이터 소스 3종

| 소스 | 용도 | 비고 |
|---|---|---|
| `suwon_features.parquet` | 학교 수·거리 (카카오 로컬 API 수집 결과) | 단지별 사전 집계 완료 |
| `suwon_complexes.parquet` | 단지 위경도 좌표 | 학원 거리 계산에 사용 |
| 소상공인 상가정보 CSV | 학원 위치 원시 데이터 | 경기도 전체 → 수원 필터 |

---
## 0. 라이브러리 임포트 & 경로 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os, re
from math import radians, sin, cos, sqrt, atan2
from difflib import get_close_matches

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

BASE      = r'C:\Users\최완우\OneDrive\Desktop\기계학습 기말 프로젝트_최한결'
FEAT_DIR  = os.path.join(BASE, 'data', 'features')
RAW_DIR   = os.path.join(BASE, 'data', 'raw')
SOHO_CSV  = r'C:\Users\최완우\OneDrive\Desktop\소상공인시장진흥공단_상가(상권)정보_경기_202512.csv'

ID_COLS = ['aptNm', 'umdNm', '_gu', '_ym', 'dealYear', 'dealMonth',
           'dealAmount', 'floor', 'excluUseAr']

print('경로 확인:')
print(f'  피처 디렉터리: {FEAT_DIR}')
print(f'  원본 디렉터리: {RAW_DIR}')
print(f'  소상공인 CSV : {SOHO_CSV}')

---
## 1. 데이터 소스 1: `suwon_features.parquet` (카카오 로컬 API 수집 결과)

### 수집 방법

학교 수·거리 데이터는 **카카오 로컬 API**를 통해 수집되었습니다.

| 항목 | 내용 |
|---|---|
| **API** | 카카오 로컬 API — 키워드 장소 검색 |
| **URL** | https://developers.kakao.com/docs/latest/ko/local/dev-guide |
| **수집 방법** | 각 단지 좌표 기준 반경 500m 내 시설 검색 |
| **카테고리** | `SC4` (초등학교), `SC5` (중학교), `SC6` (고등학교) |
| **수집 결과** | 단지별 학교 수(elem_cnt, middle_cnt, high_cnt), 최근접 거리(school_nearest_m) |

> **참고**: 카카오 로컬 API의 반경 검색 결과가 이미 `suwon_features.parquet`에 집계되어 있으므로,  
> 이 노트북에서는 해당 파일을 입력 데이터로 사용합니다.

In [ ]:
# suwon_features.parquet 로드 및 교육 관련 컬럼 추출
df_sw = pd.read_parquet(os.path.join(FEAT_DIR, 'suwon_features.parquet'))

EDU_SOURCE = ['elem_cnt', 'middle_cnt', 'high_cnt', 'school_nearest_m']
existing   = [c for c in ID_COLS + EDU_SOURCE if c in df_sw.columns]
df_e       = df_sw[existing].copy()

print(f'전체 데이터 shape  : {df_sw.shape}')
print(f'교육 컬럼 추출 후  : {df_e.shape}')
print(f'\n추출된 컬럼: {existing}')
print(f'\n--- 교육 원본 컬럼 기초 통계 ---')
df_e[EDU_SOURCE].describe().round(2)

---
## 2. `elem_cnt_500m` / `mid_cnt_500m` / `high_cnt_500m` — 학교 수 (500m 반경)

### 변수 설명

카카오 로컬 API로 각 단지 좌표 기준 **반경 500m 이내** 학교를 검색한 결과입니다.

- **초등학교(`elem_cnt_500m`)**: 주요 학군 지표 — 초등학생 자녀 가구의 수요에 직접 영향
- **중학교(`mid_cnt_500m`)**: 중학군 접근성 — 학군 배정에 민감한 가구 대상
- **고등학교(`high_cnt_500m`)**: 고교 접근성 — 학원가 밀집도와 상관

### 전처리 과정
1. 원본 컬럼명 변경: `elem_cnt` → `elem_cnt_500m` 등
2. 접근 더미 생성: 1개 이상이면 1, 없으면 0

In [ ]:
# 컬럼명 변환 및 기초 통계
df_e['elem_cnt_500m'] = df_e['elem_cnt']
df_e['mid_cnt_500m']  = df_e['middle_cnt']
df_e['high_cnt_500m'] = df_e['high_cnt']

for col, label in [('elem_cnt_500m', '초등학교'),
                   ('mid_cnt_500m',  '중학교'),
                   ('high_cnt_500m', '고등학교')]:
    s = df_e[col]
    print(f'[{label}] {col}')
    print(f'  결측={s.isna().sum()} | min={s.min():.0f} | max={s.max():.0f} | mean={s.mean():.2f}')
    print(f'  분포: {s.value_counts().sort_index().to_dict()}\n')

In [ ]:
# 접근 더미 생성 (1=반경 내 해당 학교 존재)
df_e['has_elem_500m'] = (df_e['elem_cnt_500m'] >= 1).astype(int)
df_e['has_mid_500m']  = (df_e['mid_cnt_500m']  >= 1).astype(int)
df_e['has_high_500m'] = (df_e['high_cnt_500m'] >= 1).astype(int)

print('학교 접근 더미 분포 (전체 거래 건수 기준):')
for d, label in [('has_elem_500m', '초등학교'),
                 ('has_mid_500m',  '중학교'),
                 ('has_high_500m', '고등학교')]:
    cnt = df_e[d].sum()
    pct = df_e[d].mean() * 100
    print(f'  {d} = 1 ({label} 있음): {cnt:>7,}건 ({pct:.1f}%)')

# 시각화: 학교 수 분포
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['steelblue', 'seagreen', 'tomato']
labels = ['초등학교 수 (elem_cnt_500m)', '중학교 수 (mid_cnt_500m)', '고등학교 수 (high_cnt_500m)']
cols   = ['elem_cnt_500m', 'mid_cnt_500m', 'high_cnt_500m']

for ax, col, label, color in zip(axes, cols, labels, colors):
    vc = df_e[col].value_counts().sort_index()
    ax.bar(vc.index, vc.values, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('학교 수')
    ax.set_ylabel('거래 건수')
    ax.set_xticks(vc.index)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('반경 500m 이내 학교 수 분포', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print('✅ 학교 수 변수 처리 완료')

---
## 3. `elem_nearest_m` / `log_elem_nearest_m` / `elem_access_score` — 최근접 학교 거리

### 변수 설명

카카오 로컬 API 검색 결과 중 **반경 500m 내 가장 가까운 학교까지의 거리(m)** 입니다.

- 원본(`school_nearest_m`): 반경 내 학교가 없으면 `NaN`
- 500m 초과 단지: `NaN` → **999m 패널티**로 대체 (거리 측면에서 불리함을 표현)

### 파생 변수

| 변수 | 공식 | 의미 |
|---|---|---|
| `log_elem_nearest_m` | `log(elem_nearest_m)` | 거리의 로그 변환 (우편향 보정) |
| `elem_access_score` | `100 / log(dist/80 + 2)` | 가까울수록 높은 접근성 점수 |

> `elem_access_score` 설계: 거리 80m(도보 1분) 기준에서 최대점(≈100점), 999m 패널티에서 최저점(≈37점)

In [ ]:
PENALTY_DIST = 999  # 반경 초과 시 부여할 패널티 거리 (m)

print(f'원본 school_nearest_m 결측: {df_e["school_nearest_m"].isna().sum():,}건 '
      f'({df_e["school_nearest_m"].isna().mean()*100:.1f}%)')
print('  → 이 결측치는 반경 500m 이내 학교가 없는 단지의 거래입니다.\n')

# NaN → 패널티 거리로 대체
df_e['elem_nearest_m'] = df_e['school_nearest_m'].fillna(PENALTY_DIST)

# 파생 변수 생성
df_e['log_elem_nearest_m'] = np.log(df_e['elem_nearest_m'])
df_e['elem_access_score']  = 100 / np.log(df_e['elem_nearest_m'] / 80 + 2)

print('처리 후 기초 통계:')
df_e[['elem_nearest_m', 'log_elem_nearest_m', 'elem_access_score']].describe().round(2)

In [ ]:
# 시각화: 거리 분포 및 접근성 점수 변환 개념
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) 원본 거리 분포
ax = axes[0]
valid = df_e.loc[df_e['elem_nearest_m'] < PENALTY_DIST, 'elem_nearest_m']
ax.hist(valid, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(valid.mean(), color='red', linestyle='--', label=f'평균 {valid.mean():.0f}m')
ax.set_title('최근접 학교 거리 분포\n(500m 이내 단지만)', fontsize=11)
ax.set_xlabel('거리 (m)')
ax.set_ylabel('거래 건수')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)

# 2) 전체 포함 (패널티 포함)
ax = axes[1]
ax.hist(df_e['elem_nearest_m'], bins=30, color='seagreen', alpha=0.8, edgecolor='white')
ax.axvline(PENALTY_DIST, color='red', linestyle='--', label=f'패널티({PENALTY_DIST}m)')
ax.set_title(f'최근접 학교 거리 분포\n(패널티 {PENALTY_DIST}m 포함)', fontsize=11)
ax.set_xlabel('거리 (m)')
ax.set_ylabel('거래 건수')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)

# 3) elem_access_score 변환 곡선
ax = axes[2]
dist_range = np.linspace(43, 999, 300)
score_range = 100 / np.log(dist_range / 80 + 2)
ax.plot(dist_range, score_range, color='tomato', linewidth=2)
ax.axvline(80,  color='gray', linestyle=':', alpha=0.7, label='80m (도보 1분)')
ax.axvline(500, color='gray', linestyle='--', alpha=0.7, label='500m (반경 경계)')
ax.axvline(999, color='red',  linestyle='--', alpha=0.7, label=f'{PENALTY_DIST}m (패널티)')
ax.set_title('elem_access_score 변환 곡선\n100 / log(dist/80 + 2)', fontsize=11)
ax.set_xlabel('거리 (m)')
ax.set_ylabel('접근성 점수')
ax.legend(fontsize=8)
ax.grid(linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()
print('✅ elem_nearest_m, log_elem_nearest_m, elem_access_score 처리 완료')

---
## 4. `school_density_index` — 학교 밀도 종합 인덱스

### 변수 설명

초·중·고 학교 수를 **가중 합산**한 종합 교육 환경 지표입니다.

```
school_density_index = (초등학교 수 × 3) + (중학교 수 × 2) + (고등학교 수 × 1)
```

**가중치 설계 근거:**
- 초등학교(×3): 부동산 가격과 상관이 가장 강함 (학부모 수요 직결)
- 중학교(×2): 학군 배정 제도로 인해 중간 영향
- 고등학교(×1): 비교적 광역 배정 → 상대적으로 낮은 가중치

In [ ]:
# 학교 밀도 종합 인덱스 생성
df_e['school_density_index'] = (
    df_e['elem_cnt_500m'] * 3 +
    df_e['mid_cnt_500m']  * 2 +
    df_e['high_cnt_500m'] * 1
)

print('school_density_index 기초 통계:')
print(df_e['school_density_index'].describe().round(2).to_string())
print(f'\n값 분포 (상위 10개):')
print(df_e['school_density_index'].value_counts().sort_index().to_string())

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
vc = df_e['school_density_index'].value_counts().sort_index()
ax.bar(vc.index, vc.values, color='mediumpurple', alpha=0.8, edgecolor='white')
ax.set_title('school_density_index 분포', fontsize=11)
ax.set_xlabel('인덱스 값 (초×3 + 중×2 + 고×1)')
ax.set_ylabel('거래 건수')
ax.grid(axis='y', linestyle='--', alpha=0.4)

ax = axes[1]
by_gu = df_e.groupby('_gu')['school_density_index'].mean().sort_values()
ax.barh(by_gu.index, by_gu.values, color='mediumpurple', alpha=0.8)
ax.set_title('구별 평균 school_density_index', fontsize=11)
ax.set_xlabel('평균 인덱스')
ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()
print('✅ school_density_index 처리 완료')

---
## 5. `academy_cnt_500m_t` — 반경 500m 이내 학원 수

### 데이터 소스 2: 소상공인시장진흥공단 상가(상권)정보

| 항목 | 내용 |
|---|---|
| **출처** | 소상공인시장진흥공단 공공데이터포털 |
| **URL** | https://www.data.go.kr → "소상공인시장진흥공단_상가(상권)정보" |
| **파일명** | `소상공인시장진흥공단_상가(상권)정보_경기_202512.csv` |
| **기준 시점** | 2025년 12월 |
| **수집 범위** | 경기도 전체 사업체 |
| **주요 컬럼** | 시군구명, 상권업종소분류명, 위도, 경도 |

### 전처리 과정 (4단계)
1. **수원 필터**: `시군구명`에 '수원' 포함 → 수원시 사업체만 추출
2. **학원 필터**: `상권업종소분류명`에 '학원' 또는 '교습' 포함
3. **Haversine 거리 계산**: 각 단지 좌표 ↔ 학원 좌표 간 실제 거리 계산
4. **단지명 매핑**: 단지명 정규화 후 exact match → fuzzy match (유사도 0.75 이상)

### 데이터 소스 3: `suwon_complexes.parquet` (단지 좌표)

| 항목 | 내용 |
|---|---|
| **출처** | 경기도 공동주택 정보 |
| **사용 목적** | 각 단지의 위도(lat), 경도(lon) 확보 → Haversine 거리 계산 |

In [ ]:
# ── 단계 1·2: 소상공인 CSV 로드 → 수원 학원 필터링 ──────────
df_soho = pd.read_csv(
    SOHO_CSV, encoding='utf-8-sig', low_memory=False,
    usecols=['시군구명', '상권업종소분류명', '위도', '경도']
)

print(f'경기도 전체 사업체: {len(df_soho):,}개')

df_soho_sw = df_soho[df_soho['시군구명'].str.contains('수원', na=False)]
print(f'수원시 사업체     : {len(df_soho_sw):,}개')

mask_ac = df_soho_sw['상권업종소분류명'].str.contains('학원|교습', na=False)
df_ac = df_soho_sw[mask_ac].copy()
print(f'수원 학원·교습    : {len(df_ac):,}개')

df_ac = df_ac.dropna(subset=['위도', '경도']).reset_index(drop=True)
print(f'좌표 유효 학원    : {len(df_ac):,}개  (결측 제거 후)\n')

print('학원 업종 소분류 분포:')
print(df_ac['상권업종소분류명'].value_counts().to_string())

In [ ]:
# ── 단계 3: Haversine 거리 계산으로 단지별 500m 내 학원 수 산출 ──
def haversine(lat1, lon1, lat2, lon2):
    """두 좌표 간 지표면 거리 (m) 계산"""
    R = 6_371_000
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlam = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1) * cos(phi2) * sin(dlam/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# 단지 좌표 로드
df_cx = pd.read_parquet(os.path.join(RAW_DIR, 'gg_housing', 'suwon_complexes.parquet'))
df_cx = df_cx.dropna(subset=['lat', 'lon']).reset_index(drop=True)
print(f'단지 수 (좌표 유효): {len(df_cx)}개')
print(df_cx[['complex_name', 'lat', 'lon']].head(3).to_string(index=False))

# 학원 좌표 배열 (반복 계산 최적화)
ac_lats = df_ac['위도'].values
ac_lons = df_ac['경도'].values

# 단지별 500m 이내 학원 수 계산
print(f'\n단지별 학원 거리 계산 중... (총 {len(df_cx)}개 단지 × {len(df_ac):,}개 학원)')
cnt_map = {}
for _, apt in df_cx.iterrows():
    dists = np.array([haversine(apt['lat'], apt['lon'], alat, alon)
                      for alat, alon in zip(ac_lats, ac_lons)])
    cnt_map[apt['complex_name']] = int((dists <= 500).sum())

counts = list(cnt_map.values())
print(f'\n단지별 500m 이내 학원 수 통계:')
print(f'  평균: {np.mean(counts):.1f}개   중앙값: {np.median(counts):.0f}개')
print(f'  최소: {np.min(counts)}개   최대: {np.max(counts)}개')
print(f'  학원 0개 단지: {sum(c == 0 for c in counts)}개  /  1개 이상: {sum(c >= 1 for c in counts)}개')

In [ ]:
# ── 단계 4: 단지명 매핑 (거래 데이터 aptNm ↔ cnt_map 키) ──────
def normalize(s):
    """단지명 정규화: 공백·특수문자 제거 + 소문자"""
    if pd.isna(s): return ''
    return re.sub(r'[\s\-_·\(\)（）\[\]]', '', str(s)).lower()

cnt_dict = {normalize(k): v for k, v in cnt_map.items()}
cnt_keys  = list(cnt_dict.keys())

def lookup_cnt(name, cutoff=0.75):
    """exact match 후 없으면 fuzzy match (유사도 cutoff 이상)"""
    norm = normalize(name)
    if norm in cnt_dict:
        return cnt_dict[norm], 'exact'
    matches = get_close_matches(norm, cnt_keys, n=1, cutoff=cutoff)
    if matches:
        return cnt_dict[matches[0]], 'fuzzy'
    return None, 'miss'

unique_apts = df_e['aptNm'].dropna().unique()
apt_cnt_map = {}
log = {'exact': 0, 'fuzzy': 0, 'miss': 0}

for name in unique_apts:
    val, how = lookup_cnt(name)
    apt_cnt_map[name] = val
    log[how] += 1

print(f'단지명 매핑 결과 (총 {len(unique_apts)}개 고유 단지):')
print(f'  ✅ exact match : {log["exact"]}개  (완전 일치)')
print(f'  🔍 fuzzy match : {log["fuzzy"]}개  (유사 단지명 매칭)')
print(f'  ❌ miss        : {log["miss"]}개  (매칭 실패 → 결측 처리)')

df_e['academy_cnt_500m_t'] = df_e['aptNm'].map(apt_cnt_map)
miss = df_e['academy_cnt_500m_t'].isna().sum()
print(f'\n매핑 후 결측: {miss:,}건 ({miss / len(df_e) * 100:.1f}%)')

In [ ]:
# 결측 처리: 구×연도 중앙값 → 구 중앙값 → 전체 중앙값
df_e['academy_cnt_500m_t'] = df_e.groupby(['_gu', 'dealYear'])['academy_cnt_500m_t'].transform(
    lambda x: x.fillna(x.median()))
df_e['academy_cnt_500m_t'] = df_e.groupby('_gu')['academy_cnt_500m_t'].transform(
    lambda x: x.fillna(x.median()))
df_e['academy_cnt_500m_t'] = df_e['academy_cnt_500m_t'].fillna(df_e['academy_cnt_500m_t'].median())
df_e['academy_cnt_500m_t'] = df_e['academy_cnt_500m_t'].round().astype(int)

print(f'결측 처리 후: {df_e["academy_cnt_500m_t"].isna().sum()}건')
print(f'\nacademy_cnt_500m_t 기초 통계:')
print(df_e['academy_cnt_500m_t'].describe().round(1).to_string())

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(df_e['academy_cnt_500m_t'], bins=40, color='darkorange', alpha=0.8, edgecolor='white')
ax.set_title('academy_cnt_500m_t 분포', fontsize=11)
ax.set_xlabel('학원 수 (500m 반경)')
ax.set_ylabel('거래 건수')
ax.axvline(df_e['academy_cnt_500m_t'].median(), color='red', linestyle='--',
           label=f'중앙값: {df_e["academy_cnt_500m_t"].median():.0f}개')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.4)

ax = axes[1]
by_gu = df_e.groupby('_gu')['academy_cnt_500m_t'].median().sort_values()
ax.barh(by_gu.index, by_gu.values, color='darkorange', alpha=0.8)
ax.set_title('구별 중앙값 (academy_cnt_500m_t)', fontsize=11)
ax.set_xlabel('학원 수 중앙값')
ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()
print('✅ academy_cnt_500m_t 처리 완료')

---
## 6. 최종 검증 & 저장

In [ ]:
MODEL_FEATS = [
    'elem_cnt_500m', 'mid_cnt_500m', 'high_cnt_500m',
    'has_elem_500m', 'has_mid_500m', 'has_high_500m',
    'elem_nearest_m', 'log_elem_nearest_m', 'elem_access_score',
    'school_density_index', 'academy_cnt_500m_t',
]

print('=' * 55)
print('  모델 투입 교육·학군 변수 11개 최종 결측 확인')
print('=' * 55)
for col in MODEL_FEATS:
    n   = df_e[col].isna().sum()
    pct = n / len(df_e) * 100
    status = '✅' if n == 0 else '⚠️'
    print(f'  {status} {col:<25}: {n}건 ({pct:.1f}%)')

print(f'\n--- 11개 변수 기초 통계 ---')
df_e[MODEL_FEATS].describe().round(2)

In [ ]:
# 저장할 컬럼 목록 구성
EDU_FEATURES = ID_COLS + [
    'elem_cnt_500m',     'mid_cnt_500m',      'high_cnt_500m',
    'has_elem_500m',     'has_mid_500m',       'has_high_500m',
    'elem_nearest_m',    'log_elem_nearest_m', 'elem_access_score',
    'school_density_index', 'academy_cnt_500m_t',
]
feat_cols = [c for c in EDU_FEATURES if c in df_e.columns]
feat_cols = list(dict.fromkeys(feat_cols))

# 저장
out_path = os.path.join(FEAT_DIR, 'edu_features.parquet')
df_e[feat_cols].to_parquet(out_path, index=False)

print(f'저장 완료: {out_path}')
print(f'shape    : {df_e[feat_cols].shape}')

# 검증
df_check = pd.read_parquet(out_path)
print(f'\n[저장 파일 검증]')
print(f'  로드 shape    : {df_check.shape}')
print(f'  결측 없음 확인: {df_check[MODEL_FEATS].isna().sum().sum() == 0}')
print('\n✅ edu_features.parquet 저장 완료')

---
## 7. 전체 파이프라인 요약

```
[데이터 소스 1] suwon_features.parquet        [데이터 소스 2] 소상공인 상가정보 CSV
  (카카오 로컬 API 수집 결과)                    (경기도 전체 사업체)
  └─ elem_cnt, middle_cnt, high_cnt               └─ 수원 필터 → 학원·교습 필터
  └─ school_nearest_m                             └─ 좌표 결측 제거
        │                                               │
        ▼                                               ▼
  [컬럼명 변환]                         [데이터 소스 3] suwon_complexes.parquet
  elem_cnt_500m / mid_cnt_500m                  (단지별 위경도)
  high_cnt_500m                                        │
        │                                              ▼
        ├── has_elem_500m / has_mid_500m     [Haversine 거리 계산]
        │   has_high_500m (더미 생성)          단지별 500m 이내 학원 수
        │                                              │
        ├── school_density_index             [단지명 매핑]
        │   (초×3 + 중×2 + 고×1)               exact match → fuzzy match
        │                                              │
        ├── elem_nearest_m (NaN→999 패널티)   [결측 처리]
        ├── log_elem_nearest_m               구×연도 중앙값 → 구 중앙값 → 전체
        └── elem_access_score                          │
                      │                               │
                      └──────────────┬────────────────┘
                                     ▼
                        [edu_features.parquet]
                        (11개 교육·학군 변수)
```

| 변수 | 수집 출처 | 역할 |
|---|---|---|
| `elem_cnt_500m` | 카카오 로컬 API | 초등학교 접근성 (핵심 학군 지표) |
| `mid_cnt_500m` | 카카오 로컬 API | 중학교 접근성 |
| `high_cnt_500m` | 카카오 로컬 API | 고등학교 접근성 |
| `has_elem/mid/high_500m` | 파생 (더미) | 학교 유무 이진 신호 |
| `elem_nearest_m` | 카카오 로컬 API | 학교까지 도보 거리 |
| `log_elem_nearest_m` | 파생 (로그) | 거리 우편향 보정 |
| `elem_access_score` | 파생 (점수화) | 접근성 연속 점수 |
| `school_density_index` | 파생 (가중합) | 교육 환경 종합 지표 |
| `academy_cnt_500m_t` | 소상공인 CSV + Haversine | 학원가 밀집도 |